# Chapter 4 — The matched filter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/04_matched_filter.ipynb)

**Goal.** A satellite's stars occupy a narrow region of the CMD (Chapter 2), while field stars spread over all of
it. Selecting or *weighting* stars by how much more likely they are under the satellite's CMD than under the
field's raises the contrast of the satellite. This is the **matched filter**, used for the tidal tails of Pal 5
(Rockosi et al. 2002, https://ui.adsabs.harvard.edu/abs/2002AJ....124..349R) and in most satellite searches since.

**You will learn**
1. three ways to select stars: a colour–magnitude box, an isochrone mask and continuous weights $w = s/b$;
2. a signal-to-noise statistic for weighted counts, with the background variance included;
3. to measure the gain of a filter on **injected** satellites, rather than on the object used to design it;
4. to apply the same filter to a stream: the tidal tails of Pal 5.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter                        # smoothing of CMD histograms and sky maps

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of survey catalogues, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the SQL that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    # Flat coordinates around the target: over a few degrees the distortion is small, and unlike (ra, dec)
    # they do not stretch the sky by 1/cos(dec), so circles on the sky stay circles in (xi, eta).
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)   # cos(angular distance)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


lvdb = load('lvdb').set_index('key')        # LVDB v1.1.1 rows: ra, dec [deg], rhalf [arcmin], distance_modulus
iso = load('isochrones_sdss')               # PARSEC isochrones, SDSS absolute magnitudes, one row per initial mass

# CMD grid shared by all filters: colour (g-r)_0 and magnitude g_0
# 40 colour cells of 0.04 mag over -0.4 < (g-r)_0 < 1.2 and 65 magnitude cells of 0.1 mag over 16 < g_0 < 22.5
# (22.5 is the faint limit of the Pal 5 file, g < 22.5, and is used for Boötes I too). cb, mb are cell edges,
# cc, mm cell centres; CC, MM have shape (40, 65): axis 0 is colour, axis 1 is magnitude.
cb, mb = np.arange(-0.4, 1.21, 0.04), np.arange(16, 22.51, 0.1)
cc, mm = 0.5 * (cb[1:] + cb[:-1]), 0.5 * (mb[1:] + mb[:-1])
CC, MM = np.meshgrid(cc, mm, indexing='ij')


def cmd_index(c, m):
    """Grid cell of each star; -1 outside the grid.
    The flat index i * len(mm) + j matches the C-order ravel() of a (len(cc), len(mm)) array such as CC."""
    i, j = np.digitize(c, cb) - 1, np.digitize(m, mb) - 1      # colour and magnitude cell numbers
    ok = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm))
    return np.where(ok, i * len(mm) + j, -1)


def satellite_cmd(mu, logage=10.1, feh=-2.1, err=lambda m: 0.02 + 0.08 * np.exp((m - 22) / 1.0), intrinsic=0.03):
    """Normalised satellite CMD density on the grid: isochrone at distance modulus mu, sampled uniformly along
    its length, weighted by the IMF, blurred by photometric errors (colour error as a function of g) and an
    intrinsic width. The default error law approximates SDSS (Chapter 0).

    Parameters: mu [mag]; logage = log10(age/yr) (10.1 = 12.6 Gyr); feh = [Fe/H]; err(g) = 1-sigma error of a
    single band [mag] (0.1 mag at g = 22, rising e-fold per mag); intrinsic = extra colour width [mag].
    Returns an array of shape (len(cc), len(mm)) that sums to 1: the fraction of satellite stars in each cell."""
    # old, metal-poor isochrone; stage <= 7 drops the last point of the AGB (stage 8, a single point in this grid);
    # sorting by initial mass orders the points along the sequence, from the lower main sequence to the AGB
    I = iso[np.isclose(iso.logage, logage) & np.isclose(iso.feh, feh) & (iso.stage <= 7)].sort_values('m_ini')
    c, g = (I.g - I.r).to_numpy(), I.g.to_numpy() + mu        # colour and apparent g of each isochrone point
    arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(c), np.diff(g)))])   # path length along it [mag]
    # 5000 points equally spaced in path length, so that fast phases (giant branch, horizontal branch), which
    # have few isochrone points, are sampled as finely as the main sequence
    s = np.linspace(0, arc[-1], 5000)
    # w: number of stars per sample point = increment of the cumulative IMF (int_imf) between neighbouring points
    pc, pg, w = np.interp(s, arc, c), np.interp(s, arc, g), np.gradient(np.interp(s, arc, I.int_imf))
    sc = np.hypot(np.sqrt(2) * err(pg), intrinsic)   # colour width: g and r errors (taken equal) + intrinsic
    sm = np.hypot(err(pg), 0.1)                      # magnitude width: g error + 0.1 mag (one magnitude cell)
    S = np.zeros(CC.shape)
    # add a normalised 2-d Gaussian per sample point, weighted by w; in chunks of 500 points so that the
    # temporary array (500, 40, 65) stays small
    for k in range(0, len(s), 500):
        sl = slice(k, k + 500)
        S += np.einsum('k,kij->ij', w[sl], np.exp(-0.5 * ((CC - pc[sl, None, None]) / sc[sl, None, None])**2
                                                      - 0.5 * ((MM - pg[sl, None, None]) / sm[sl, None, None])**2)
                       / sc[sl, None, None] / sm[sl, None, None])
    return S / S.sum()


def field_cmd(c, m, smooth=1.0):
    """Normalised field CMD density on the grid, from the stars of a control region (smoothed histogram).
    The density is floored at one star per cell: where the control region has no stars we do not know b, and
    an unfloored s/b would put enormous weight on (nearly) empty cells.

    smooth: Gaussian width in cells (1 cell = 0.04 mag in colour, 0.1 mag in g) to reduce Poisson noise."""
    H = gaussian_filter(np.histogram2d(c, m, [cb, mb])[0], smooth)   # smoothed counts, shape (len(cc), len(mm))
    return np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()

## 4.1 Three filters for Boötes I

For each star $i$ we define a weight $w_i$ from its position in the CMD:

- **box**: $w = 1$ inside $0 < (g-r)_0 < 0.7$, $19 < g_0 < 22.5$ (the selection of Chapter 0), else 0;
- **isochrone mask**: $w = 1$ where the satellite density $s$ is above a threshold enclosing 90% of the satellite's
  stars, else 0;
- **matched filter**: $w = s/b$, the ratio of the satellite and field CMD densities (normalised to a maximum of 1).

The statistic is the weighted excess over the background, divided by its expected noise. With $N$ stars in an
on-region and a background predicted from an off-region of $\alpha^{-1}$ times its area,

$$ S = \sum_{\rm on} w_i - \alpha\sum_{\rm off} w_i, \qquad \sigma^2 \simeq \alpha\sum_{\rm off} w_i^2, \qquad {\rm S/N} = S/\sigma. $$

The variance is that of a weighted Poisson sum (each background star contributes $w_i^2$); the off-region term is
neglected because the off-region is ~100 times larger. The weights matter through the ratio of the signal they
keep to the $\sqrt{\sum w^2}$ of the background they admit — which is why $w = s/b$ is optimal for a weak signal.
One practical detail: $b$ is estimated from a finite control sample, so in sparsely populated parts of the CMD it is
poorly known; we floor it at one control star per cell, which caps the weights there (see `field_cmd`).

In [ ]:
ref = lvdb.loc['bootes_1']
cat = load('bootes1_sdss')                       # SDSS DR9 point sources, g < 23, within 2° of Boötes I
# extinction-corrected magnitude and colour (the file stores observed magnitudes and the SFD98 extinction)
g0 = (cat.psfmag_g - cat.extinction_g).to_numpy()
c0 = g0 - (cat.psfmag_r - cat.extinction_r).to_numpy()
R = np.hypot(*tangent(cat.ra, cat.dec, ref.ra, ref.dec)) * 60   # distance from the LVDB centre [arcmin]
# control (off) region: the annulus 60'-120', i.e. beyond 6 r_h of Boötes I (r_h = 10') and out to the
# 2° edge of the file; it defines the field CMD and predicts the background
off = (R > 60) & (R < 120)
k_star = cmd_index(c0, g0)                       # CMD cell of every star (-1 outside the grid)

S_sat = satellite_cmd(ref.distance_modulus)      # satellite model at the LVDB distance modulus, 19.11
B_fld = field_cmd(c0[off], g0[off])              # field model from the control annulus
# isochrone-mask threshold: sort the cell densities in decreasing order and take the density at which their
# cumulative sum first reaches 0.9, so that cells with s >= level hold 90% of the satellite's stars
level = np.sort(S_sat.ravel())[::-1][np.searchsorted(np.cumsum(np.sort(S_sat.ravel())[::-1]), 0.9)]
# Each filter is an array of weights on the CMD grid, shape (len(cc), len(mm)).
# The S/N below does not change if all weights are multiplied by a constant, so s/b may be scaled to a maximum of 1.
filters = {
    'box': ((CC > 0) & (CC < 0.7) & (MM > 19) & (MM < 22.5)).astype(float),      # the box of Chapter 0
    'isochrone mask': (S_sat >= level).astype(float),
    'matched filter s/b': (S_sat / B_fld) / (S_sat / B_fld).max(),
}


def weights(F, k):
    """Weight of each star (index k into the CMD grid) under filter F.
    Stars outside the grid (k = -1) get weight 0; np.maximum(k, 0) only avoids indexing with -1."""
    return np.where(k >= 0, F.ravel()[np.maximum(k, 0)], 0.0)


def snr(w, on, off, alpha):
    """S/N of the weighted excess in the on-region: (sum_on w - alpha sum_off w) / sqrt(alpha sum_off w^2).
    w: weight of every star; on, off: boolean selections; alpha: area(on) / area(off)."""
    return (w[on].sum() - alpha * w[off].sum()) / np.sqrt(alpha * (w[off]**2).sum())


on = R < ref.rhalf                               # on-region: inside the LVDB half-light radius, 9.97'
alpha = ref.rhalf**2 / (120**2 - 60**2)          # area ratio on/off = pi r_h^2 / pi (120^2 - 60^2), arcmin^2
for name, F in filters.items():
    print(f'{name:20s}: S/N of Boötes I inside r_h = {snr(weights(F, k_star), on, off, alpha):5.1f}')

# Figure 4.1: s, b, the isochrone mask and s/b on the CMD grid (the box is not shown)
fig, ax = plt.subplots(1, 4, figsize=(17, 4.2), sharey=True)
ext = [cb[0], cb[-1], mb[-1], mb[0]]             # imshow extent: colour on x, g_0 increasing downwards
for a, (t, F) in zip(ax, [('satellite s (μ = 19.11)', S_sat), ('field b', B_fld)] + list(filters.items())[1:]):
    a.imshow(F.T**0.3, extent=ext, aspect='auto', cmap='Greys')            # power 0.3 shows faint features too
    a.set(title=t, xlabel='$(g-r)_0$')
ax[0].set(ylabel='$g_0$')
plt.tight_layout()

**Figure 4.1.** The four CMD filters on the common grid ($(g-r)_0$ from −0.4 to 1.2 in 0.04-mag cells,
horizontal; $g_0$ from 16 to 22.5 in 0.1-mag cells, vertical, bright at the top); grey scale is value$^{0.3}$, darker =
larger, each panel scaled to its own maximum. From left to right: (1) the satellite density $s$, a PARSEC isochrone
of 12.6 Gyr and [Fe/H] = −2.1 at μ = 19.11, weighted by the IMF and blurred by the SDSS error model — the blue
horizontal branch (BHB) at $g_0 \approx 19.6$, $(g-r)_0 \approx -0.2$ to 0.2, the giant branch rising to
$g_0 \approx 17$, and the subgiant branch at the faint limit ($g_0 \approx 22.2$–22.5); (2) the field density $b$
from the SDSS point sources (extinction-corrected) in the annulus 60′–120′ around Boötes I, smoothed by one cell and
floored at one star per cell: a ridge at $(g-r)_0 \approx 0.2$–0.5 at all magnitudes and a broader population redwards;
(3) the isochrone mask (black, value 1), the cells enclosing 90% of $s$: the BHB, the giant branch and, at
$g_0 \gtrsim 20.4$, the full colour range, because there $s$ is smeared by the large errors; (4) the matched-filter
weights $s/b$, normalised to a maximum of 1. The weights are largest on the BHB, where satellite stars are
expected and field stars are rare, and next along the blue edge of the giant branch. Inside $r_h$ of Boötes I the
printed S/N is 17.9 (box), 20.0 (isochrone mask) and 48.7 (matched filter).

## 4.2 Measuring the gain on injected satellites

Comparing the filters on Boötes I itself is biased: the box was chosen by looking at Boötes I. A fair test injects
synthetic satellites whose stars are drawn from the satellite CMD model into the *real* field, and compares the
filters on them. We place 200 satellites at random positions in the annulus 60′–120′ (away from Boötes I), each with
40 stars inside a 10′ radius, uniformly distributed in that circle, and compute the S/N of each filter using the real
field stars in the same circle plus the injected ones. The background is predicted from the rest of the annulus.

In [ ]:
rng = np.random.default_rng(4)                   # fixed seed: the injections are reproducible
xi, eta = tangent(cat.ra, cat.dec, ref.ra, ref.dec)   # star positions [deg] about the Boötes I centre
pcell = S_sat.ravel() / S_sat.sum()              # probability of each CMD cell for a satellite star
# 40 stars per injected satellite, all inside an aperture of radius 10' (r_ap in deg; comparable to r_h of Boötes I)
n_inj, r_ap = 40, 10 / 60
gains = {k: [] for k in filters}                 # S/N of every injection, per filter
for t in range(200):                             # 200 injections (the number fixed in the plan, STRUCTURE.md)
    rad, ang = rng.uniform(1.25, 1.75), rng.uniform(0, 2 * np.pi)          # centre well inside the annulus
    # (the 10' aperture then spans radii 1.08°-1.92°, entirely inside the 1°-2° control annulus)
    x0, y0 = rad * np.cos(ang), rad * np.sin(ang)
    on_t = np.hypot(xi - x0, eta - y0) < r_ap   # real field stars in the aperture
    off_t = off & ~on_t                          # background: the rest of the annulus
    alpha_t = (np.pi * r_ap**2) / (np.pi * (2**2 - 1**2) - np.pi * r_ap**2)   # area ratio aperture / rest [deg^2]
    # Only the CMD cells of the injected stars matter: inside the aperture their positions do not enter the counts
    k_inj = rng.choice(pcell.size, n_inj, p=pcell)                           # CMD cells of the injected stars
    sn = {}
    for name, F in filters.items():
        w_real = weights(F, k_star)
        # weighted counts in the aperture (real + injected) minus the background predicted from the annulus
        signal = w_real[on_t].sum() + F.ravel()[k_inj].sum() - alpha_t * w_real[off_t].sum()
        sn[name] = signal / np.sqrt(alpha_t * (w_real[off_t]**2).sum())     # noise: background term only
    for name in filters:
        gains[name].append(sn[name])
sn_arr = {k: np.array(v) for k, v in gains.items()}       # filter -> S/N of the 200 injections
gain = sn_arr['matched filter s/b'] / sn_arr['box']       # per-injection ratio of S/N, same stars
# 95% interval on the mean gain: 2000 bootstrap resamples of the 200 injections
boot = np.array([rng.choice(gain, gain.size).mean() for _ in range(2000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print({k: round(float(np.median(v)), 1) for k, v in sn_arr.items()})
print(f'gain of matched filter over box: mean {gain.mean():.2f}, 95% interval [{lo:.2f}, {hi:.2f}]')

# Figure 4.2: histograms of S/N, bins of 0.5 from -2 to 14
fig, ax = plt.subplots(figsize=(6.5, 4))
for name, v in sn_arr.items():
    ax.hist(v, np.linspace(-2, 14, 33), histtype='step', lw=2, label=f'{name}: median {np.median(v):.1f}')
ax.set(xlabel='S/N of an injected 40-star satellite', ylabel='injections')
ax.legend(fontsize=8)
plt.tight_layout()
# test: the matched filter beats the box on average, with the whole 95% bootstrap interval of the mean gain above 1
assert lo > 1

**Figure 4.2.** Histograms (bins of 0.5) of the S/N of 200 synthetic satellites, each of 40 stars drawn from the
model CMD $s$ of Figure 4.1 and added to the real SDSS stars inside a 10′ aperture placed at random in the annulus
60′–120′ around Boötes I; the background is predicted from the rest of the annulus. Blue: box; orange: isochrone
mask; green: matched filter $s/b$. Medians 2.3, 2.9 and 7.5; the mean per-injection gain of the matched filter
over the box is 3.63, 95% bootstrap interval [3.09, 4.15]. The isochrone mask beats the box, and the continuous
weights beat both, on the same stars; the matched-filter S/N also scatters much more (≈1–14).

The gain is not a free lunch. Most of it comes from the BHB: the
PARSEC model puts all horizontal-branch stars of an old, metal-poor population on the blue side, as in Boötes I,
but satellites with red horizontal branches exist, and for them the gain is smaller. In general the gain holds only
if the satellite resembles the assumed isochrone and distance; a search must therefore scan the distance
(Chapter 5) and, ideally, the population.

## 4.3 The same filter on a stream: Pal 5

The globular cluster Pal 5 (μ = 16.7, ~22 kpc) is being torn apart by the Milky Way's tides; its stars form two
tails more than 20° long (Odenkirchen et al. 2001; Rockosi et al. 2002). Pal 5 is younger and more metal-rich than
Boötes I, so we use log age = 10.05 and [Fe/H] = −1.3. We weight every star of the 22° × 16° field by $s/b$, with $b$
from the whole field (the stream is a tiny fraction of it), and map the summed weights in 0.1° cells, divided by
the fraction of each cell that SDSS observed.

In [ ]:
p5 = lvdb.loc['palomar_05']
pal = load('pal5_sdss')                          # SDSS point sources, RA 222°-244°, Dec -8°...+8°, g < 22.5
cov = load('pal5_coverage')                      # counts of all SDSS objects (r < 22) in 0.1° x 0.1° cells
pg0 = (pal.psfmag_g - pal.extinction_g).to_numpy()            # extinction-corrected g and g - r
pc0 = pg0 - (pal.psfmag_r - pal.extinction_r).to_numpy()
kp = cmd_index(pc0, pg0)
S5 = satellite_cmd(p5.distance_modulus, logage=10.05, feh=-1.3)   # Pal 5: mu = 16.71, 11.2 Gyr, [Fe/H] = -1.3
B5 = field_cmd(pc0, pg0)                         # field from all stars of the field (the stream is a tiny part)
# box: 0.1 < (g-r)_0 < 0.5, 19.5 < g_0 < 22.5, around the main-sequence turn-off of Pal 5
# (g_0 = 20.9, (g-r)_0 = 0.24 for this isochrone at mu = 16.71) and the upper main sequence
F5 = {'box': ((CC > 0.1) & (CC < 0.5) & (MM > 19.5) & (MM < 22.5)).astype(float),
      'matched filter s/b': (S5 / B5) / (S5 / B5).max()}
ra_e, dec_e = np.arange(222, 244.01, 0.1), np.arange(-8, 8.01, 0.1)   # 0.1° map cells: 220 x 160
# Footprint: a map cell counts as observed if SDSS has more than 20 objects in it (a fully observed cell has
# ~140, the median of cov.n), so partly covered edge cells are excluded
observed = np.zeros((len(ra_e) - 1, len(dec_e) - 1), bool)
i, j = cov.ira - 2220, cov.idec + 80             # ira = floor(ra / 0.1), idec = floor(dec / 0.1) -> map indices
ok = (i >= 0) & (i < observed.shape[0]) & (j >= 0) & (j < observed.shape[1]) & (cov.n > 20)
observed[i[ok], j[ok]] = True

fig, ax = plt.subplots(1, 2, figsize=(15, 5.6))
maps = {}
for a, (name, F) in zip(ax, F5.items()):
    w = weights(F, kp)
    H = np.histogram2d(pal.ra, pal.dec, [ra_e, dec_e], weights=w)[0]   # summed weights per 0.1° cell
    # Gaussian smoothing with sigma = 1.5 cells (0.15°), divided by the equally smoothed footprint so that cells
    # near the edge of the footprint are not biased low; the floor 1e-3 avoids division by zero; unobserved -> NaN
    H = np.where(observed, gaussian_filter(H, 1.5) / np.maximum(gaussian_filter(observed.astype(float), 1.5), 1e-3), np.nan)
    maps[name] = H
    v = np.nanpercentile(H, [2, 99.5])           # grey scale from the 2nd to the 99.5th percentile
    a.imshow(H.T, origin='lower', extent=[ra_e[0], ra_e[-1], dec_e[0], dec_e[-1]], cmap='Greys', vmin=v[0], vmax=v[1])
    a.set(xlabel='RA [deg]', ylabel='Dec [deg]', title=name, xlim=(244, 222))   # RA increases to the left (east)
plt.tight_layout()

**Figure 4.3.** Maps of the Pal 5 field (SDSS point sources, RA 222°–244°, Dec −8°…+8°, extinction-corrected),
RA increasing to the left. Left: number of stars per 0.1° cell in the box $0.1 < (g-r)_0 < 0.5$,
$19.5 < g_0 < 22.5$; right: sum of the matched-filter weights $s/b$ per cell, with $s$ the isochrone of 11.2 Gyr,
[Fe/H] = −1.3 at μ = 16.71 and $b$ the CMD of the whole field. Both maps are smoothed with a Gaussian of
σ = 0.15° and divided by the equally smoothed footprint; grey scale from the 2nd to the 99.5th percentile, darker =
more stars. White: outside the SDSS footprint (cells with ≤ 20 SDSS objects), including the region south of
Dec ≈ −3° and the rectangle at RA 240°–244°, Dec ≈ 1.3°–2.6°; the stripes at the bottom right are partly covered
scans. Pal 5 sits at (229.0, −0.1); its two tails run north-east to RA ≈ 234° and south-west to RA ≈ 226°, and are
sharper with the matched filter. The second blob, at (229.6, 2.1), is the globular cluster M5 (7.5 kpc; the white
pixels at its centre are cells of its crowded core that contain no SDSS objects): its stars are brighter than Pal 5's, but its main
sequence crosses the filter. The dark eastern (left) edge, strongest in the north-east corner, is the rising density of
the Galactic disc and bulge towards lower latitude. Chapter 7 measures the stream's track and significance.

## Summary

- The S/N of a weighted count is $S/\sqrt{\alpha\sum_{\rm off} w^2}$; weights $s/b$ maximise it for a faint signal.
- Judge a filter on injected objects, not on the one that inspired it.
- The same filter reveals the tidal tails of Pal 5, a stream spanning the whole field.